# Multi-Layer Perceptron (MLP)

This model is a neural network written from scratch in plain Python lists. It predicts the policy class of each *province* (`INCREASE = 1`, `RETAIN = 0`, `DECREASE = -1`) from its standardized features.

Architecture: **input → hidden (ReLU) → output (softmax, 3 classes)**, trained with batch gradient descent.

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import random
from collections import Counter
from statistics import mean, stdev
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from os import path

from methods.read_data import createMatrix, printMatrix
from methods.normalize_data import standardizeData, coalesceData
from models.mlp import gradient_descent, make_predictions, get_accuracy
from evaluation.holdout_validation import holdout_split, holdout_validation
from evaluation.kfold_cross_validation import kFoldSplit, crossValidation, accuracy
from evaluation.conf_matrix import evaluate

### Step 1: Create the Dataset from the `PROJECT_DATASET` folder
- This will create a matrix with 12 fields/attributes and 87 provinces

In [3]:
data_set_dir = path.join(path.abspath("."), "PROJECT_DATASET")          # Get the path to the dataset folder
provinces, fields, matrix = createMatrix(Path(data_set_dir))            # Create the matrix from the dataset
printMatrix(provinces, fields, matrix)                                  # Print the matrix with all of the provinces and fields

87 provinces x 12 fields
showing fields 0 to 11 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6 | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Table_7 | Information and Communication
  [9] DERIVED | Change in poverty incidence 2021 to 2023 (percentage points)
  [10] DERIVED | Change in poverty incidence 2018 to 2023 (percentage points)
  [11] DERIVED | Income inequality (top 10% share / bottom 40% share)

Province                       [0]         [1]         [2]         [3]         [4]         [5]         [6]         [7]         [8]         [9]        [10]        [11]
NCR 1st District               2.4         1.7         3.0         0.1       46.54     37711.0      482.79

### Step 2: Cleanup the data
- Coalesce missing values to 0, then standardize it using Z-score Scaling
- `featuresList` is a list of feature vectors, in the same order as `provinces`

In [4]:
# Coalesce NA values to 0, then standardize it.
cleaned_data = standardizeData(coalesceData(matrix))
featuresList = [list(row) for row in cleaned_data]      # Plain lists, so the label can be appended later

print(f"{len(provinces)} provinces, {len(featuresList[0])} features each")

87 provinces, 12 features each


### Step 3: Load the labels from `province_policy_results.csv`
- Unlike k-means, the MLP is *supervised*, so every province needs its true label
- Read the actual class of each province from the tagged dataset, the same way as in the k-means notebook
- The tagged dataset is created by `methods/label_data.py` from the policies in the Policy for Labeling Data section of the project document
- The class is converted to `INCREASE = 1`, `RETAIN = 0`, `DECREASE = -1`
- `labelList` follows the order of `provinces`, so `featuresList[i]` and `labelList[i]` belong to the same province

In [5]:
# Read the actual class of each province from the tagged dataset
tagged_dir = path.join(data_set_dir, "province_policy_results.csv")    # Get the path to the tagged dataset
with open(tagged_dir, newline="", encoding="utf-8-sig") as file:
  actual_classes = {row["Province"]: row["Class"] for row in csv.DictReader(file)}

# Convert the class of each province to a number, in the same order as provinces
CLASS_TO_LABEL = {"INCREASE": 1, "RETAIN": 0, "DECREASE": -1}
labelList = [CLASS_TO_LABEL[actual_classes[id]] for id in provinces]

print("Label counts:", Counter(labelList))

Label counts: Counter({1: 46, 0: 31, -1: 10})


### Step 4: Map the labels to class indices
- The output layer has 3 neurons (one per class), so the network works with the indices `0, 1, 2`, not `-1, 0, 1`
- `CLASSES` is the single source of truth: `index[label]` going in, `CLASSES[i]` coming out
- A raw `-1` would silently land in the last column through negative indexing, so always map first

In [6]:
CLASSES = (-1, 0, 1)                                          # Column order of the output layer
index = {label: i for i, label in enumerate(CLASSES)}         # {-1: 0, 0: 1, 1: 2}
print("Label -> index:", index)

Label -> index: {-1: 0, 0: 1, 1: 2}


### Step 5: Split the provinces into a training set and a testing set
- 80% of the provinces are used for training and the remaining 20% for testing
- The training rows carry the true label as the last element, so it is stripped off before training

In [7]:
training_list, testing_rows, test_labels, test_indices = holdout_split(featuresList, labelList, 0.8)

X_train = [row[:-1] for row in training_list]                 # Features only
Y_train = [index[row[-1]] for row in training_list]           # Labels as class indices
print(f"Training provinces: {len(X_train)}, Testing provinces: {len(testing_rows)}")

Training provinces: 70, Testing provinces: 17


### Step 6: Train the network
- Hidden layer of 10 neurons, learning rate `alpha = 0.1`
- Each iteration runs one forward pass over all training provinces, one backward pass (averaged gradients), and **one** weight update
- Training progress is printed every 2 iterations, so this demo uses only 20 iterations

In [16]:
random.seed(1)                                                # Same starting weights every run
w1, b1, w2, b2 = gradient_descent(X_train, Y_train, alpha=0.1, iterations=250, hiddenNum=65, outputNum=3)

Iteration:  0
Accuracy: 15.714285714285714%
Iteration:  2
Accuracy: 60.0%
Iteration:  4
Accuracy: 75.71428571428571%
Iteration:  6
Accuracy: 75.71428571428571%
Iteration:  8
Accuracy: 80.0%
Iteration:  10
Accuracy: 80.0%
Iteration:  12
Accuracy: 80.0%
Iteration:  14
Accuracy: 81.42857142857143%
Iteration:  16
Accuracy: 80.0%
Iteration:  18
Accuracy: 80.0%
Iteration:  20
Accuracy: 80.0%
Iteration:  22
Accuracy: 81.42857142857143%
Iteration:  24
Accuracy: 81.42857142857143%
Iteration:  26
Accuracy: 81.42857142857143%
Iteration:  28
Accuracy: 81.42857142857143%
Iteration:  30
Accuracy: 81.42857142857143%
Iteration:  32
Accuracy: 82.85714285714286%
Iteration:  34
Accuracy: 82.85714285714286%
Iteration:  36
Accuracy: 82.85714285714286%
Iteration:  38
Accuracy: 82.85714285714286%
Iteration:  40
Accuracy: 82.85714285714286%
Iteration:  42
Accuracy: 82.85714285714286%
Iteration:  44
Accuracy: 82.85714285714286%
Iteration:  46
Accuracy: 82.85714285714286%
Iteration:  48
Accuracy: 82.85714285714

### Step 7: Predict the provinces in the testing set
- `make_predictions` returns class indices, which are decoded back to `-1 / 0 / 1` with `CLASSES`

In [17]:
predictions = [CLASSES[p] for p in make_predictions(testing_rows, w1, b1, w2, b2)]

print(f"{'Province':<25} {'True':>5} {'Predicted':>10}")
for i, true, pred in zip(test_indices, test_labels, predictions):
    print(f"{provinces[i]:<25} {true:>5} {pred:>10}")

correct = sum(t == p for t, p in zip(test_labels, predictions))
print(f"\nTest accuracy: {correct}/{len(test_labels)} = {correct / len(test_labels):.2%}")

Province                   True  Predicted
Dinagat Islands               1          1
NCR 1st District             -1         -1
Northern Samar                1          1
Siquijor                     -1          0
NCR 4th District             -1         -1
Bukidnon                      1          1
La Union                      0          0
Batangas                     -1          0
Negros Oriental               0          1
Zamboanga Sibugay             1          1
Samar                         1          1
Camiguin                      0          0
Cagayan                       0          0
Occidental Mindoro            1          1
Kalinga                       0          0
Cotabato                      1          1
Nueva Vizcaya                 0          0

Test accuracy: 14/17 = 82.35%


### Step 8: Confusion Matrix of the Holdout Validation
- Compare the predicted class with the actual class of the testing provinces
- Compute the Recall, Precision, F1 Score and Accuracy for each class (one vs. rest)

In [18]:
# Calculate and display the confusion matrix and the performance metrics for each class
holdout_metrics = evaluate(test_labels, predictions)

Confusion Matrix (Positive class: -1)
                      Predicted    
                  Positive Negative
Actual    Positive       2        2
          Negative       0       13
Recall:    0.5000
Precision: 1.0000
F1 Score:  0.6667
Accuracy:  0.8824

Confusion Matrix (Positive class: 0)
                      Predicted    
                  Positive Negative
Actual    Positive       5        1
          Negative       2        9
Recall:    0.8333
Precision: 0.7143
F1 Score:  0.7692
Accuracy:  0.8235

Confusion Matrix (Positive class: 1)
                      Predicted    
                  Positive Negative
Actual    Positive       7        0
          Negative       1        9
Recall:    1.0000
Precision: 0.8750
F1 Score:  0.9333
Accuracy:  0.9412



### Step 9: K-Fold Cross Validation
- The provinces are split into 5 folds, each fold is the testing set once and the other folds are the training set
- `mlp_model` trains a new network on the training folds, then predicts the testing fold, with the same settings as in Step 6
- Every province is predicted exactly once, so the accuracy uses all of the provinces

In [19]:
def mlp_model(testing_rows, training_list):
    X = [row[:-1] for row in training_list]                   # Features only
    Y = [index[row[-1]] for row in training_list]             # Labels as class indices

    random.seed(1)                                            # Same starting weights every run
    with redirect_stdout(StringIO()):                         # Hide the training progress
        w1, b1, w2, b2 = gradient_descent(X, Y, alpha=0.1, iterations=20, hiddenNum=10, outputNum=3)

    predicted = make_predictions(testing_rows, w1, b1, w2, b2)
    return [testing_rows[i] + [CLASSES[predicted[i]]] for i in range(len(testing_rows))]

folds = 5
foldList = kFoldSplit(len(featuresList), folds)

# Each result is a tuple (province index, true label, predicted label, fold)
kfold_results = crossValidation(featuresList, labelList, foldList, mlp_model)

fold_accuracies = []
for f in range(folds):
    fold_results = [result for result in kfold_results if result[3] == f + 1]
    fold_accuracies.append(accuracy(fold_results))
    correct = sum(result[1] == result[2] for result in fold_results)
    print(f"Fold {f + 1}: {correct}/{len(fold_results)} = {fold_accuracies[-1]:.2%}")

correct = sum(result[1] == result[2] for result in kfold_results)
print(f"\nK-fold accuracy: {correct}/{len(kfold_results)} = {accuracy(kfold_results):.2%}")
print(f"Average of the folds: {mean(fold_accuracies):.2%}, standard deviation: {stdev(fold_accuracies):.2%}")

Fold 1: 14/18 = 77.78%
Fold 2: 10/18 = 55.56%
Fold 3: 13/17 = 76.47%
Fold 4: 12/17 = 70.59%
Fold 5: 12/17 = 70.59%

K-fold accuracy: 61/87 = 70.11%
Average of the folds: 70.20%, standard deviation: 8.82%


### Step 10: Confusion Matrix of the K-Fold Cross Validation
- The predictions from the k-fold cross validation are used, so the confusion matrix counts all of the provinces

In [20]:
actual = [result[1] for result in kfold_results]              # Actual class from the tagged dataset
predicted = [result[2] for result in kfold_results]           # Predicted class from the network

# Calculate and display the confusion matrix and the performance metrics for each class
kfold_metrics = evaluate(actual, predicted)

Confusion Matrix (Positive class: -1)
                      Predicted    
                  Positive Negative
Actual    Positive       2        8
          Negative       5       72
Recall:    0.2000
Precision: 0.2857
F1 Score:  0.2353
Accuracy:  0.8506

Confusion Matrix (Positive class: 0)
                      Predicted    
                  Positive Negative
Actual    Positive      20       11
          Negative      14       42
Recall:    0.6452
Precision: 0.5882
F1 Score:  0.6154
Accuracy:  0.7126

Confusion Matrix (Positive class: 1)
                      Predicted    
                  Positive Negative
Actual    Positive      39        7
          Negative       7       34
Recall:    0.8478
Precision: 0.8478
F1 Score:  0.8478
Accuracy:  0.8391

